# NYC Borough and Subway Station GeoJSON Collection

## Imports and Folder Setup

In [2]:
import requests
import json
import os
import zipfile
import io
import geopandas as gpd
import pandas as pd

notebooks_folder = os.getcwd() 
raw_root_folder = os.path.abspath( os.path.join(notebooks_folder, "..", "data", "raw"))
interim_root_folder = os.path.abspath( os.path.join(notebooks_folder, "..", "data", "interim"))
processed_root_folder = os.path.abspath( os.path.join(notebooks_folder, "..", "data", "processed"))

## Download NYC Borough GeoJSON

In [31]:
# Download NYC borough boundaries # Clipped to Shoreline 
borough_url = "https://services5.arcgis.com/GfwWNkhOj9bNBqoJ/arcgis/rest/services/NYC_Borough_Boundary/FeatureServer/0/query?where=1=1&outFields=*&outSR=4326&f=geojson"

response = requests.get(borough_url)
borough_geojson = response.json()

# borough_path = os.path.join(raw_root_folder, "nyc_boroughs.geojson")
borough_path = os.path.join(processed_root_folder, "New York City Boroughs.geojson")
with open(borough_path, 'w') as f:
    json.dump(borough_geojson, f)

print(f"Downloaded: {len(borough_geojson['features'])} boroughs")


Downloaded: 5 boroughs


## Download Subway Stations Data and Create GeoJSON

In [3]:
# -------------------------
# Setup paths
# -------------------------
subway_raw_folder = os.path.join(raw_root_folder, "New York City Subway Stations")
os.makedirs(subway_raw_folder, exist_ok=True)

# -------------------------
# Download MTA GTFS data
# -------------------------
gtfs_url = "http://web.mta.info/developers/data/nyct/subway/google_transit.zip"
response = requests.get(gtfs_url)

# Extract needed GTFS files from zip
with zipfile.ZipFile(io.BytesIO(response.content)) as z:
    # Stops
    with z.open('stops.txt') as f:
        stops_df = pd.read_csv(f)
    
    # Routes
    with z.open('routes.txt') as f:
        routes_df = pd.read_csv(f)
    
    # Trips
    with z.open('trips.txt') as f:
        trips_df = pd.read_csv(f)
    
    # Stop times
    with z.open('stop_times.txt') as f:
        stop_times_df = pd.read_csv(f)

# Optional: Save raw CSVs
stops_df.to_csv(os.path.join(subway_raw_folder, "stops.csv"), index=False)
routes_df.to_csv(os.path.join(subway_raw_folder, "routes.csv"), index=False)
trips_df.to_csv(os.path.join(subway_raw_folder, "trips.csv"), index=False)
stop_times_df.to_csv(os.path.join(subway_raw_folder, "stop_times.csv"), index=False)

# -------------------------
# Download and extract NYC borough shapefile
# -------------------------
nybb_url = "https://s-media.nyc.gov/agencies/dcp/assets/files/zip/data-tools/bytes/borough-boundaries/nybb_25c.zip"
nybb_zip_path = os.path.join(subway_raw_folder, "nybb_25c.zip")
nybb_extract_dir = os.path.join(subway_raw_folder, "nybb_25c")

if not os.path.exists(nybb_zip_path):
    print("Downloading borough boundaries shapefile...")
    r = requests.get(nybb_url)
    with open(nybb_zip_path, "wb") as f:
        f.write(r.content)
else:
    print("File already exists:", nybb_zip_path)

# Extract zip
with zipfile.ZipFile(nybb_zip_path, "r") as z:
    z.extractall(nybb_extract_dir)

# The zip contains a nested folder named nybb_25c
nybb_shp = os.path.join(nybb_extract_dir, "nybb_25c", "nybb.shp")
borough_gdf = gpd.read_file(nybb_shp)
borough_gdf = borough_gdf.to_crs(epsg=4326)
borough_gdf = borough_gdf[["BoroName", "geometry"]]

# -------------------------
# Filter parent stations and assign boroughs
# -------------------------
stations = stops_df[stops_df["location_type"] == 1].copy()
stations = stations[["stop_id", "stop_name", "stop_lat", "stop_lon"]]

stations_gdf = gpd.GeoDataFrame(
    stations,
    geometry=gpd.points_from_xy(stations["stop_lon"], stations["stop_lat"]),
    crs="EPSG:4326"
)

# Spatial join with boroughs
joined = gpd.sjoin(
    stations_gdf,
    borough_gdf,
    how="left",
    predicate="within"
)
stations["borough"] = joined["BoroName"].fillna("Unknown")

print("\nStations by borough:")
print(stations["borough"].value_counts(dropna=False))

# # -------------------------
# # Compute subway lines per station
# # -------------------------
# Merge stop_times -> trips -> routes to get route_short_name per platform
stop_routes = stop_times_df.merge(trips_df[['trip_id','route_id']], on='trip_id')
stop_routes = stop_routes.merge(routes_df[['route_id','route_short_name']], on='route_id')

# Merge with stops to get parent_station for each platform
stop_routes = stop_routes.merge(
    stops_df[['stop_id','parent_station']], 
    left_on='stop_id', right_on='stop_id', how='left'
)

# Only keep platforms that have a parent_station
platform_routes = stop_routes[stop_routes['parent_station'].notna()]

# Group by parent_station to get all unique lines for that station
station_lines = (
    platform_routes.groupby('parent_station')['route_short_name']
    .apply(lambda x: sorted(set(x)))  # unique & sorted
    .reset_index()
    .rename(columns={'parent_station':'stop_id','route_short_name':'lines'})
)

# Merge the lines into your stations (parent stations) dataframe
stations = stations.merge(station_lines, on='stop_id', how='left')

# -------------------------
# Create GeoJSON
# -------------------------
subway_geojson = {
    "type": "FeatureCollection",
    "features": []
}

for _, row in stations.iterrows():
    subway_geojson["features"].append({
        "type": "Feature",
        "geometry": {
            "type": "Point",
            "coordinates": [row["stop_lon"], row["stop_lat"]]
        },
        "properties": {
            "name": row["stop_name"],
            "stop_id": row["stop_id"],
            "borough": row["borough"],
            "lines": row["lines"]
        }
    })

output_path = os.path.join(processed_root_folder, "New York City Subway Stations.geojson")
with open(output_path, "w") as f:
    json.dump(subway_geojson, f, indent=2)

print("\nSaved GeoJSON with lines:", output_path)


File already exists: /Users/KailenMcCauley/Documents/Georgia Tech/CSE 6242/NYC-Affordability-Transit-Access/data/raw/New York City Subway Stations/nybb_25c.zip

Stations by borough:
borough
Brooklyn         169
Manhattan        153
Queens            83
Bronx             70
Staten Island     21
Name: count, dtype: int64

Saved GeoJSON with lines: /Users/KailenMcCauley/Documents/Georgia Tech/CSE 6242/NYC-Affordability-Transit-Access/data/processed/New York City Subway Stations.geojson


## Reoganizing Sales Data into GeoJSON

In [55]:
all_sales = []

nyc_sales_data_path = os.path.join(interim_root_folder, "New York City Sales Data")
for year in os.listdir(nyc_sales_data_path):
    year_folder = os.path.join(nyc_sales_data_path, year)
    if not os.path.isdir(year_folder):
        continue

    for file in os.listdir(year_folder):
        if not file.lower().endswith((".xlsx")) or file.startswith("~"):
            continue

        file_path = os.path.join(year_folder, file)
        df = pd.read_excel(file_path)
        df["SALE_YEAR"] = pd.to_datetime(df["SALE DATE"]).dt.year
        df["BOROUGH_FILE"] = file.removesuffix(".xlsx") 
        all_sales.append(df)

sales_df = pd.concat(all_sales, ignore_index=True)
print("Total sales rows:", len(sales_df))
print(sales_df.head())

# Create GeoDataFrame
houses_gdf = gpd.GeoDataFrame(
    sales_df,
    geometry=gpd.points_from_xy(sales_df["LON"], sales_df["LAT"]),
    crs="EPSG:4326"
)

# Clean up SALE_DATE as datetime
houses_gdf["SALE_DATE"] = pd.to_datetime(houses_gdf["SALE DATE"], errors="coerce")

# Attach borough polygons
houses_gdf = gpd.sjoin(
    houses_gdf,
    borough_gdf[["BoroName", "geometry"]],
    how="left",
    predicate="within"
).rename(columns={"BoroName": "borough"})

if "index_right" in houses_gdf.columns:
    houses_gdf = houses_gdf.drop(columns=["index_right"])

# Keep only the columns relevant for your map
houses_for_geojson = houses_gdf[[
    "FULL ADDRESS",
    "borough",
    "LAT",
    "LON",
    "SALE PRICE",
    "SALE DATE",
    "SALE_YEAR",
    "geometry"
]].rename(columns={
    "FULL ADDRESS": "address"
})

# Light cleaning before writing
g = houses_for_geojson.copy()
g = g[g.geometry.notnull()]
g = g[g.geometry.is_valid]
g = g[
    (g.geometry.x.between(-80, -70)) &
    (g.geometry.y.between(35, 45))
]

# Ensure types are JSON-friendly
g["SALE DATE"] = pd.to_datetime(g["SALE DATE"], errors="coerce").dt.strftime("%Y-%m-%d")
g["SALE PRICE"] = pd.to_numeric(g["SALE PRICE"], errors="coerce")

houses_geojson_path = os.path.join(processed_root_folder, "New York City Houses.geojson")
g = g.rename(columns={
    "SALE PRICE": "SALE_PRICE",
    "SALE DATE": "SALE_DATE"
})
g.to_file(houses_geojson_path, driver="GeoJSON")
print("Saved houses GeoJSON:", houses_geojson_path)

Total sales rows: 474736
                                        FULL ADDRESS  SALE PRICE  SALE DATE  \
0    65 Avenue D, Alphabet City, New York, NY, 10009     1560000 2013-07-18   
1  243 7Th Street, Alphabet City, New York, NY, 1...     3150000 2013-03-06   
2  238 4Th Street, Alphabet City, New York, NY, 1...     3450000 2013-03-27   
3    17 Avenue B, Alphabet City, New York, NY, 10009         283 2013-04-18   
4    14 Avenue B, Alphabet City, New York, NY, 10009    13185684 2013-01-31   

         LAT        LON  SALE_YEAR BOROUGH_FILE  
0  40.720314 -73.864704       2013    Manhattan  
1  40.672668 -73.990010       2013    Manhattan  
2  40.888990 -73.912779       2013    Manhattan  
3  40.783730 -73.908361       2013    Manhattan  
4  40.786004 -73.850281       2013    Manhattan  
Saved houses GeoJSON: /Users/KailenMcCauley/Documents/Georgia Tech/CSE 6242/NYC-Affordability-Transit-Access/data/processed/New York City Houses.geojson
